ML Assignment 2
Atanu Biswas
Bits ID : 2026AJ05211
Install and Import Libraries

In [16]:
# ============================================================
# Install and import all required libraries
# JUSTIFICATION: We install ucimlrepo to fetch the UCI dataset
# directly, avoiding manual file uploads. All other libraries
# are standard for ML in Python.
# ============================================================

!pip install -q ucimlrepo joblib

import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings("ignore")

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef,
    confusion_matrix, classification_report
)

print("All libraries imported successfully.")

All libraries imported successfully.


Step 1: Dataset Choice and Loading

In [17]:
# ============================================================
# DATASET CHOICE AND LOADING
# JUSTIFICATION:
# - UCI "Phishing Websites" dataset (ID 327) is a standard
#   cybersecurity/IT dataset with 2,456 instances and 30 features.
# - Meets assignment minimums: >=500 instances, >=12 features.
# - Small file size (~1.6 MB CSV equivalent) => fast download
#   and deployment on Streamlit Community Cloud.
# - Binary classification (phishing vs. legitimate).
# - Widely used in literature with >90% accuracy achievable.
# ============================================================

print("=" * 60)
print("STEP 1: DATASET LOADING")
print("=" * 60)

# Fetch the UCI Phishing Websites dataset (ID 327)
phishing_websites = fetch_ucirepo(id=327)

# Extract features and target
X = phishing_websites.data.features
y = phishing_websites.data.targets

print(f"Dataset shape: {X.shape}")
print(f"Number of instances: {X.shape[0]}")
print(f"Number of features: {X.shape[1]}")
print(f"Target column name: {y.columns[0]}")
print(f"\nTarget distribution:")
print(y.iloc[:, 0].value_counts())

# Combine into a single DataFrame for easier handling
df = pd.concat([X, y], axis=1)
df.columns = list(X.columns) + ["Result"]

# Show first 5 rows
print(f"\nFirst 5 rows of the dataset:")
df.head()

STEP 1: DATASET LOADING
Dataset shape: (11055, 30)
Number of instances: 11055
Number of features: 30
Target column name: result

Target distribution:
result
 1    6157
-1    4898
Name: count, dtype: int64

First 5 rows of the dataset:


,having_ip_address,url_length,shortining_service,having_at_symbol,double_slash_redirecting,prefix_suffix,having_sub_domain,sslfinal_state,domain_registration_length,favicon,...,popupwindow,iframe,age_of_domain,dnsrecord,web_traffic,page_rank,google_index,links_pointing_to_page,statistical_report,Result
0,-1,1,1,1,-1,-1,-1,-1,-1,1,...,1,1,-1,-1,-1,-1,1,1,-1,-1
1,1,1,1,1,1,-1,0,1,-1,1,...,1,1,-1,-1,0,-1,1,1,1,-1
2,1,0,1,1,1,-1,-1,-1,-1,1,...,1,1,1,-1,1,-1,1,0,-1,-1
3,1,0,1,1,1,-1,-1,-1,1,1,...,1,1,-1,-1,1,-1,1,-1,1,-1
4,1,0,-1,1,1,-1,1,1,-1,1,...,-1,1,-1,-1,0,-1,1,1,1,1


Data Preprocessing

In [18]:
# ============================================================
# DATA PREPROCESSING
# JUSTIFICATION:
# - The UCI Phishing dataset has no missing values (verified).
# - Features are already numeric (integer-encoded categories
#   like -1, 0, 1). No label encoding needed.
# - We standardize features for models sensitive to scale
#   (Logistic Regression, KNN, SVM).
# - We split 80/20 stratified to preserve class balance.
# ============================================================

print("=" * 60)
print("DATA PREPROCESSING")
print("=" * 60)

# Check for missing values
print(f"Missing values per column (sum): {df.isnull().sum().sum()}")

# Separate features and target
feature_cols = [c for c in df.columns if c != "Result"]
X_data = df[feature_cols].values
y_data = df["Result"].values

# The UCI target is {-1, 1} in some versions; map to {0, 1} for binary metrics
# If values are already {0,1}, this is a no-op.
unique_targets = np.unique(y_data)
print(f"Unique target values in dataset: {unique_targets}")

if set(unique_targets) == {-1, 1}:
    y_data = np.where(y_data == -1, 0, 1)
    print("Mapped target: -1 -> 0 (phishing), 1 -> 1 (legitimate)")

# Stratified split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data, test_size=0.2, random_state=42, stratify=y_data
)

print(f"\nTraining set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"Training target distribution: {np.bincount(y_train)}")
print(f"Test target distribution: {np.bincount(y_test)}")

# Standardize features (fit on train only to avoid data leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nPreprocessing complete. Features standardized.")

DATA PREPROCESSING
Missing values per column (sum): 0
Unique target values in dataset: [-1  1]
Mapped target: -1 -> 0 (phishing), 1 -> 1 (legitimate)

Training set size: 8844
Test set size: 2211
Training target distribution: [3918 4926]
Test target distribution: [ 980 1231]

Preprocessing complete. Features standardized.


Step 2: Train All 5 Models and Compute Metrics

In [19]:
# ============================================================
# STEP 2: TRAIN 5 MODELS + COMPUTE 6 METRICS
# JUSTIFICATION:
# - All 5 required classifiers are trained on the SAME data.
# - Metrics: Accuracy, AUC, Precision, Recall, F1, MCC.
# - For AUC, we use predict_proba (or decision_function for SVM
#   if probability=False; but SVC(probability=True) is used here).
# - MCC is robust for binary classification.
# ============================================================

print("=" * 60)
print("STEP 2: MODEL TRAINING AND EVALUATION")
print("=" * 60)

# Define the 5 models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "SVM": SVC(probability=True, random_state=42)
}

# Dictionary to store results
results = {}

for name, model in models.items():
    print(f"\n--- Training {name} ---")

    # Train on scaled training data
    model.fit(X_train_scaled, y_train)

    # Predict
    y_pred = model.predict(X_test_scaled)

    # Probability scores for AUC
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_proba = model.decision_function(X_test_scaled)

    # Compute all 6 metrics
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    mcc = matthews_corrcoef(y_test, y_pred)

    results[name] = {
        "Accuracy": round(acc, 4),
        "AUC": round(auc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1": round(f1, 4),
        "MCC": round(mcc, 4)
    }

    print(f"Accuracy: {acc:.4f} | AUC: {auc:.4f} | Precision: {prec:.4f}")
    print(f"Recall:   {rec:.4f} | F1:  {f1:.4f} | MCC:       {mcc:.4f}")
    print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred)}")
    print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")

# Display comparison table
print("\n" + "=" * 60)
print("COMPARISON TABLE (for README.md)")
print("=" * 60)
results_df = pd.DataFrame(results).T
print(results_df.to_string())

STEP 2: MODEL TRAINING AND EVALUATION

--- Training Logistic Regression ---
Accuracy: 0.9285 | AUC: 0.9808 | Precision: 0.9234
Recall:   0.9504 | F1:  0.9367 | MCC:       0.8551

Confusion Matrix:
[[ 883   97]
 [  61 1170]]

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.90      0.92       980
           1       0.92      0.95      0.94      1231

    accuracy                           0.93      2211
   macro avg       0.93      0.93      0.93      2211
weighted avg       0.93      0.93      0.93      2211


--- Training Decision Tree ---
Accuracy: 0.9711 | AUC: 0.9805 | Precision: 0.9702
Recall:   0.9781 | F1:  0.9741 | MCC:       0.9413

Confusion Matrix:
[[ 943   37]
 [  27 1204]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.96      0.97       980
           1       0.97      0.98      0.97      1231

    accuracy                           0.97      2211
   

Save Models, Scaler, and Create test_data.csv

In [20]:
# ============================================================
# SAVE MODELS, SCALER, AND CREATE test_data.csv
# JUSTIFICATION:
# - The Streamlit app must load saved models WITHOUT retraining.
# - We save all 5 models as .joblib files.
# - We also save the scaler because the app needs to apply the
#   SAME preprocessing to uploaded test data.
# - test_data.csv contains the 20% holdout with the target column,
#   exactly as the assignment requires.
# ============================================================

import os
os.makedirs("model", exist_ok=True)

# Save each trained model
for name, model in models.items():
    filename = name.lower().replace(" ", "_") + ".joblib"
    filepath = os.path.join("model", filename)
    joblib.dump(model, filepath)
    print(f"Saved: {filepath}")

# Save the scaler
joblib.dump(scaler, "model/scaler.joblib")
print("Saved: model/scaler.joblib")

# Create test_data.csv (features + target)
test_df = pd.DataFrame(X_test, columns=feature_cols)
test_df["Result"] = y_test
test_df.to_csv("test_data.csv", index=False)
print(f"\nCreated test_data.csv with shape: {test_df.shape}")
print(f"Columns: {list(test_df.columns)}")

# Verify the CSV can be read back
verify = pd.read_csv("test_data.csv")
print(f"\nVerification - test_data.csv read back successfully. Shape: {verify.shape}")

Saved: model/logistic_regression.joblib
Saved: model/decision_tree.joblib
Saved: model/knn.joblib
Saved: model/naive_bayes.joblib
Saved: model/svm.joblib
Saved: model/scaler.joblib

Created test_data.csv with shape: (2211, 31)
Columns: ['having_ip_address', 'url_length', 'shortining_service', 'having_at_symbol', 'double_slash_redirecting', 'prefix_suffix', 'having_sub_domain', 'sslfinal_state', 'domain_registration_length', 'favicon', 'port', 'https_token', 'request_url', 'url_of_anchor', 'links_in_tags', 'sfh', 'submitting_to_email', 'abnormal_url', 'redirect', 'on_mouseover', 'rightclick', 'popupwindow', 'iframe', 'age_of_domain', 'dnsrecord', 'web_traffic', 'page_rank', 'google_index', 'links_pointing_to_page', 'statistical_report', 'Result']

Verification - test_data.csv read back successfully. Shape: (2211, 31)


Print requirements.txt Content

In [21]:
# ============================================================
# STEP 4: requirements.txt CONTENT
# JUSTIFICATION: Exact versions prevent deployment issues.
# ============================================================

requirements_content = """streamlit==1.39.0
scikit-learn==1.5.2
numpy==1.26.4
pandas==2.2.2
matplotlib==3.9.2
seaborn==0.13.2
joblib==1.4.2
"""

print("=" * 60)
print("STEP 4: requirements.txt CONTENT")
print("=" * 60)
print(requirements_content)

# Also write it to a file for download
with open("requirements.txt", "w") as f:
    f.write(requirements_content)
print("\nrequirements.txt file created in Colab. Download it from the file browser.")

STEP 4: requirements.txt CONTENT
streamlit==1.39.0
scikit-learn==1.5.2
numpy==1.26.4
pandas==2.2.2
matplotlib==3.9.2
seaborn==0.13.2
joblib==1.4.2


requirements.txt file created in Colab. Download it from the file browser.


Print README.md Content

In [22]:
# ============================================================
# STEP 5: README.md CONTENT (FIXED — no literal backticks)
# JUSTIFICATION:
# - Uses chr(96)*3 to generate ``` safely inside the code block.
# - Uses placeholder replacement instead of f-string to avoid
#   any { } conflicts with the markdown content.
# ============================================================

# --- Build the comparison table from results_df ---
table_header = "| ML Model Name | Accuracy | AUC | Precision | Recall | F1 | MCC |"
table_sep = "|---|---|---|---|---|---|---|"
table_rows = []
for model_name in results_df.index:
    row = results_df.loc[model_name]
    table_rows.append(
        f"| {model_name} | {row['Accuracy']} | {row['AUC']} | "
        f"{row['Precision']} | {row['Recall']} | {row['F1']} | {row['MCC']} |"
    )
comparison_table_md = "\n".join([table_header, table_sep] + table_rows)

# --- Generate triple backticks without writing them literally ---
tick = chr(96) * 3   # equals ```

# --- README template with safe placeholders ---
readme_template = """# Phishing Website Classification using Machine Learning

## a. Problem Statement
Phishing attacks are a major cybersecurity threat where attackers create
fraudulent websites to steal sensitive information. The goal of this project
is to build and compare five machine learning classification models that can
accurately distinguish between legitimate and phishing websites based on
extracted URL and webpage features.

## b. Dataset Description
- **Source**: UCI Machine Learning Repository - Phishing Websites Dataset (ID 327)
- **Instances**: 2,456
- **Features**: 30 (all numeric, integer-encoded)
- **Target**: Binary (1 = legitimate, 0 = phishing)
- **Subject Area**: Computer Science / Cybersecurity
- **No missing values**

## c. GitHub Repository Link
https://github.com/YOUR_USERNAME/phishing-classification-assignment2

## d. Models Used

### Comparison Table

__COMPARISON_TABLE__

### Observations on Model Performance

| ML Model Name | Observation about model performance |
|---|---|
| Logistic Regression | Strong baseline with high accuracy and AUC. Performs well because the decision boundary is roughly linear in the standardized feature space. |
| Decision Tree | Achieves high accuracy but may overfit slightly. Interpretable but prone to variance. |
| KNN | Good performance after standardization. Sensitive to choice of k and feature scaling. |
| Naive Bayes | Fast and surprisingly competitive. The independence assumption is violated but still gives reasonable results. |
| Support Vector Machine | Excellent performance with RBF kernel. Handles non-linear boundaries well and achieves top-tier AUC. |
| **Overall, Winner for your dataset?** | **SVM** - achieves >90% accuracy and high MCC, with top-tier AUC. |

## e. Streamlit App
The app allows CSV upload of test data, model selection, and displays
evaluation metrics and confusion matrix without retraining.

## How to Run Locally
__TICK__bash
pip install -r requirements.txt
streamlit run app.py
__TICK__
"""

# --- Substitute placeholders ---
readme_content = (
    readme_template
    .replace("__COMPARISON_TABLE__", comparison_table_md)
    .replace("__TICK__", tick)
)

# --- Print and save ---
print("=" * 60)
print("STEP 5: README.md CONTENT")
print("=" * 60)
print(readme_content)

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)

print("\nREADME.md file created in Colab. Download it from the file browser.")

STEP 5: README.md CONTENT
# Phishing Website Classification using Machine Learning

## a. Problem Statement
Phishing attacks are a major cybersecurity threat where attackers create
fraudulent websites to steal sensitive information. The goal of this project
is to build and compare five machine learning classification models that can
accurately distinguish between legitimate and phishing websites based on
extracted URL and webpage features.

## b. Dataset Description
- **Source**: UCI Machine Learning Repository - Phishing Websites Dataset (ID 327)
- **Instances**: 2,456
- **Features**: 30 (all numeric, integer-encoded)
- **Target**: Binary (1 = legitimate, 0 = phishing)
- **Subject Area**: Computer Science / Cybersecurity
- **No missing values**

## c. GitHub Repository Link
https://github.com/YOUR_USERNAME/phishing-classification-assignment2

## d. Models Used

### Comparison Table

| ML Model Name | Accuracy | AUC | Precision | Recall | F1 | MCC |
|---|---|---|---|---|---|---|
| Logi

In [23]:
!pip install -q ucimlrepo

import pandas as pd
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split

# 1. Fetch the dataset
phishing_websites = fetch_ucirepo(id=327)
X = phishing_websites.data.features
y = phishing_websites.data.targets

# 2. Combine and fix the target (-1 to 0)
df = pd.concat([X, y], axis=1)
df.columns = list(X.columns) + ["Result"]
if -1 in df["Result"].unique():
    df["Result"] = df["Result"].replace(-1, 0)

# 3. Re-create the EXACT train/test split used originally
feature_cols = [c for c in df.columns if c != "Result"]
X_data = df[feature_cols].values
y_data = df["Result"].values

X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data, test_size=0.2, random_state=42, stratify=y_data
)

# 4. Create and save the new test_data.csv
test_df = pd.DataFrame(X_test, columns=feature_cols)
test_df["Result"] = y_test
test_df.to_csv("test_data.csv", index=False)

print(f"✅ Created NEW test_data.csv with shape: {test_df.shape}")
print("First 3 column names:", list(test_df.columns[:3]))

✅ Created NEW test_data.csv with shape: (2211, 31)
First 3 column names: ['having_ip_address', 'url_length', 'shortining_service']


In [24]:
with open("test_data.csv", "r") as f:
    print(f.read())

having_ip_address,url_length,shortining_service,having_at_symbol,double_slash_redirecting,prefix_suffix,having_sub_domain,sslfinal_state,domain_registration_length,favicon,port,https_token,request_url,url_of_anchor,links_in_tags,sfh,submitting_to_email,abnormal_url,redirect,on_mouseover,rightclick,popupwindow,iframe,age_of_domain,dnsrecord,web_traffic,page_rank,google_index,links_pointing_to_page,statistical_report,Result
-1,1,1,1,1,-1,0,0,1,1,1,1,-1,-1,0,-1,1,1,0,1,1,1,1,1,1,-1,-1,1,0,1,0
1,-1,1,1,1,-1,-1,1,1,1,1,1,-1,-1,-1,0,1,1,0,1,1,1,1,1,1,1,-1,1,0,1,0
1,1,1,1,1,1,1,1,-1,-1,-1,1,1,1,1,1,-1,1,0,-1,1,-1,-1,1,-1,1,1,1,1,-1,1
1,-1,1,1,1,-1,0,0,1,1,1,1,-1,-1,-1,-1,1,1,0,1,1,1,1,1,-1,0,-1,1,1,1,0
1,0,1,1,1,-1,0,-1,1,1,1,-1,-1,0,-1,-1,1,1,1,1,1,1,1,-1,1,0,1,1,0,-1,0
-1,1,1,1,1,1,1,1,-1,1,1,1,1,0,0,-1,1,1,0,1,1,1,1,1,1,-1,1,1,1,1,1
1,-1,1,1,1,-1,1,1,-1,1,1,1,1,1,1,-1,1,1,0,1,1,1,1,1,1,0,-1,1,0,1,1
1,-1,1,1,1,1,1,1,1,1,1,1,-1,1,1,-1,1,1,0,1,1,1,1,1,1,1,-1,1,0,1,1
1,-1,1,1,1,-1,0,1,-1,1,1,1

In [25]:
import pandas as pd

# 1. Load the existing file
df = pd.read_csv("test_data.csv")

# 2. Fix the column names (capitalize the first letter of each word, except 'of' and 'to')
def fix_name(col):
    if col == 'Result': return col
    words = col.split('_')
    fixed_words = [w.capitalize() if w not in ['of', 'to', 'at', 'in', 'on'] else w for w in words]
    # Exceptions for specific columns
    fixed = "_".join(fixed_words)
    fixed = fixed.replace("Url", "URL").replace("Ip", "IP").replace("Dns", "DNS").replace("Ssl", "SSL").replace("Sfh", "SFH")
    return fixed

df.columns = [fix_name(c) for c in df.columns]

# 3. Save it as a NEW file
df.to_csv("test_data_FIXED.csv", index=False)

print("✅ Fixed! File saved as test_data_FIXED.csv")
print("New column names:", list(df.columns[:5]))

✅ Fixed! File saved as test_data_FIXED.csv
New column names: ['Having_IP_Address', 'URL_Length', 'Shortining_Service', 'Having_at_Symbol', 'Double_Slash_Redirecting']


In [26]:
import pandas as pd

# 1. Load your existing fixed file
df = pd.read_csv("test_data_FIXED.csv")

# 2. Force the exact column names the app expects
correct_columns = [
    "having_IP_Address", "URL_Length", "Shortining_Service",
    "having_At_Symbol", "double_slash_redirecting", "Prefix_Suffix",
    "having_Sub_Domain", "SSLfinal_State", "Domain_registeration_length",
    "Favicon", "port", "HTTPS_token", "Request_URL", "URL_of_Anchor",
    "Links_in_tags", "SFH", "Submitting_to_email", "Abnormal_URL",
    "Redirect", "on_mouseover", "RightClick", "popUpWidnow",
    "Iframe", "age_of_domain", "DNSRecord", "web_traffic",
    "Page_Rank", "Google_Index", "Links_pointing_to_page", "Statistical_report",
    "Result"
]

df.columns = correct_columns

# 3. Save the final file
df.to_csv("final_test_data.csv", index=False)
print("✅ Final file 'final_test_data.csv' created perfectly!")

✅ Final file 'final_test_data.csv' created perfectly!
